# Dataset match × ratings d'équipe SoFIFA

Une ligne = un match **entre deux des 91 équipes notées**. Pour chaque équipe (domicile / extérieur) :
`overall`, `attack`, `midfield`, `defence`, `build_up_style`, `defensive_line`, `defensive_approach`. Plus le score
et le résultat.

**Trois briques, aucune ne rouvre le navigateur :**
1. **Features d'équipe** — re-parsées depuis le HTML SoFIFA déjà sauvegardé (`data/raw/sofifa/...`), avec le parseur
   **corrigé** (la colonne `defence` du CSV était buggée : elle valait la ligne défensive, pas la vraie note).
2. **Pont `team_id` ↔ SoFIFA** — via le **club modal des joueurs** (`player.club_name`, même source que SoFIFA) :
   fiable en correspondance exacte, contrairement au fuzzy sur les noms d'équipe (qui confondait AS Roma / AS Monaco).
3. **Matchs** — depuis `match_team`, filtrés à la **saison 2025-26** (proche du snapshot ratings, cf. `SEASON_CUTOFF`)
   et à ceux où **les deux équipes ont un rating**.

In [1]:
from pathlib import Path
import re, json

import pandas as pd
from bs4 import BeautifulSoup
from unidecode import unidecode
from sqlalchemy import create_engine, text

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)

PROJECT_ROOT = Path.cwd()
RAW_HTML_DIR = PROJECT_ROOT / "data" / "raw" / "sofifa" / "team_profiles"
# CSV utilisé UNIQUEMENT pour les noms (id -> sofifa_team_name / uefa_club_name).
# Les NOTES viennent du HTML : la colonne `defence` est buggée dans les deux CSV.
RATINGS_CSV = next((p for p in [
    PROJECT_ROOT / "data" / "sofifa_team_profiles_enriched.csv",
    PROJECT_ROOT / "data" / "sofifa_team_profiles_full.csv",
] if p.exists()), None)
OUTPUT_CSV = PROJECT_ROOT / "data" / "match_outcome_team_ratings.csv"

# Option 1 : ne garder que les matchs proches du snapshot ratings (FC 26, saison 2025-26).
SEASON_CUTOFF = "2025-07-01"

FEATURES = ["overall", "attack", "midfield", "defence",
            "build_up_style", "defensive_line", "defensive_approach"]


def load_env_file(path):
    values = {}
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            values[k.strip()] = v.strip().strip('"').strip("'")
    return values


env = load_env_file(PROJECT_ROOT / ".env")
url = (env.get("NEON_DATABASE_URL") or env.get("DATABASE_URL") or env.get("POSTGRES_URL") or "")
url = url.replace("postgresql://", "postgresql+psycopg2://", 1).replace("postgres://", "postgresql+psycopg2://", 1)
engine = create_engine(url, pool_pre_ping=True)


def norm(s):
    s = unidecode(str(s)).lower()
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()


print("HTML dir :", RAW_HTML_DIR, "| existe :", RAW_HTML_DIR.exists())
print("ratings  :", RATINGS_CSV)

HTML dir : /Users/safidyramaherison/Documents/Jedha/AIFS/Projects/Final_project/Futbol/github/football-predictor/data/raw/sofifa/team_profiles | existe : True
ratings  : /Users/safidyramaherison/Documents/Jedha/AIFS/Projects/Final_project/Futbol/github/football-predictor/data/sofifa_team_profiles_enriched.csv


## 1. Features d'équipe depuis le HTML (parseur corrigé)

Sur la page SoFIFA, les 4 notes sont rendues **valeur puis label** (`84 · Overall · 86 · Attack · 84 · Midfield ·
84 · Defence`). Le bug d'origine : le label `def` matchait *aussi* « **Def**ensive approach » (le réglage tactique)
et récupérait sa valeur. On corrige en matchant le **label exact** et en prenant le nombre **juste avant**.

In [2]:
RATING_LABELS = {"overall": "overall", "attack": "attack", "midfield": "midfield",
                 "defence": "defence", "defense": "defence"}


def html_lines(path):
    soup = BeautifulSoup(path.read_text(encoding="utf-8"), "lxml")
    return [" ".join(s.split()) for s in soup.stripped_strings if s.split()]


def parse_ratings(lines):
    # label EXACT, valeur = ligne précédente. Exclut "Defensive approach".
    out = {}
    for i, line in enumerate(lines):
        key = RATING_LABELS.get(line.strip().lower())
        if key and key not in out and i > 0 and lines[i - 1].strip().isdigit():
            out[key] = int(lines[i - 1].strip())
    return out


def parse_tactics(lines):
    out = {"build_up_style": "", "defensive_line": "", "defensive_approach": ""}
    for i, line in enumerate(lines):
        low = line.strip().lower()
        if low == "build-up style" and i + 1 < len(lines):
            out["build_up_style"] = lines[i + 1].strip()
        if low == "defensive approach" and i + 2 < len(lines):
            out["defensive_line"] = lines[i + 1].strip()
            out["defensive_approach"] = lines[i + 2].strip()
    return out


rows = []
for html_path in sorted(RAW_HTML_DIR.rglob("*.html")):
    sofifa_id = int(html_path.stem)
    lines = html_lines(html_path)
    rows.append({"sofifa_team_id": sofifa_id, **parse_ratings(lines), **parse_tactics(lines)})

parsed = pd.DataFrame(rows)
parsed["defensive_line"] = pd.to_numeric(parsed["defensive_line"], errors="coerce")

# noms depuis le CSV existant (par id)
names = pd.read_csv(RATINGS_CSV, dtype=str)[["sofifa_team_id", "sofifa_team_name", "uefa_club_name"]]
names["sofifa_team_id"] = names["sofifa_team_id"].astype(int)
team_features = parsed.merge(names, on="sofifa_team_id", how="left")

print("features d'équipe :", team_features.shape)
print("\ndéfense — désormais continue (fini le 30/50/65/90) :")
print(team_features["defence"].describe().round(1).to_string())
display(team_features[["sofifa_team_name", *FEATURES]].head(10))

features d'équipe : (91, 10)

défense — désormais continue (fini le 30/50/65/90) :
count    91.0
mean     75.2
std       4.7
min      67.0
25%      72.0
50%      75.0
75%      79.0
max      85.0


,sofifa_team_name,overall,attack,midfield,defence,build_up_style,defensive_line,defensive_approach
0,Arsenal,84,86,84,84,Balanced,65,High
1,Manchester City,84,82,85,82,Balanced,65,High
2,FC Lugano,70,71,68,68,Counter,50,Balanced
3,FCSB,69,69,70,68,Balanced,30,Deep
4,Shakhtar Donetsk,72,71,73,73,Balanced,50,Balanced
5,Manchester United,80,82,80,79,Balanced,65,High
6,Fiorentina,77,79,75,75,Short passing,65,High
7,Viktoria Plzeň,72,73,71,73,Short passing,90,High
8,Jagiellonia Białystok,69,72,69,69,Balanced,50,Balanced
9,RB Leipzig,78,77,76,80,Counter,65,High


## 2. Pont `team_id` (base) ↔ équipe SoFIFA

Le `team_name` de la base (« Associazione Sportiva Roma ») ne matche pas les noms SoFIFA. On passe par le **club
modal des joueurs** de chaque `team_id` : `player.club_name` vient de SoFIFA, donc la correspondance est **exacte**.

In [3]:
q_modal = text('''
    SELECT l.team_id, p.club_name, COUNT(*) AS n
    FROM "public"."lineup" l
    JOIN "public"."player" p ON p.player_id = l.player_id
    WHERE p.club_name IS NOT NULL
    GROUP BY l.team_id, p.club_name
''')
with engine.connect() as conn:
    club_counts = pd.read_sql(q_modal, conn)
    teams = pd.read_sql(text('SELECT team_id, team_name FROM "public"."team"'), conn)

modal = (club_counts.sort_values("n", ascending=False)
         .drop_duplicates("team_id")
         .rename(columns={"club_name": "modal_club"}))
teams = teams.merge(modal[["team_id", "modal_club"]], on="team_id", how="left")

# clé normalisée du club modal -> sofifa_team_id
feat_key = {norm(n): i for n, i in zip(team_features["sofifa_team_name"], team_features["sofifa_team_id"])}
teams["k"] = teams["modal_club"].map(norm)
teams["sofifa_team_id"] = teams["k"].map(feat_key)

# corrections manuelles éventuelles : team_id -> sofifa_team_id
TEAM_OVERRIDES = {}
for tid, sid in TEAM_OVERRIDES.items():
    teams.loc[teams["team_id"] == tid, "sofifa_team_id"] = sid

team_to_sofifa = dict(zip(teams.loc[teams["sofifa_team_id"].notna(), "team_id"],
                          teams.loc[teams["sofifa_team_id"].notna(), "sofifa_team_id"].astype(int)))
print(f"team_id reliés à un rating : {len(team_to_sofifa)}/{len(teams)}")

reached = set(team_to_sofifa.values())
unreached = team_features[~team_features["sofifa_team_id"].isin(reached)]
print(f"ratings jamais atteints par un team_id : {len(unreached)}")
display(unreached[["sofifa_team_id", "sofifa_team_name", "uefa_club_name"]])

team_id reliés à un rating : 89/99
ratings jamais atteints par un team_id : 2


,sofifa_team_id,sofifa_team_name,uefa_club_name
43,245,Ajax,Ajax
51,278,AEK Athens,AEK Athens


## 3. Matchs + résultat

`match_team` donne le score de chaque côté. On construit le résultat comme **libellé** (`home_win` / `draw` /
`away_win`) — pas de codage 1/0/-1 (le nul n'est pas « à mi-chemin »). On garde ensuite **les matchs où les deux
équipes ont un rating**.

In [4]:
q_mt = text('''
    SELECT mt.match_id, mt.team_id, mt.side, mt.score, m.match_date
    FROM "public"."match_team" mt
    JOIN "public"."match" m ON m.match_id = mt.match_id
''')
with engine.connect() as conn:
    mt = pd.read_sql(q_mt, conn)
mt["score"] = pd.to_numeric(mt["score"], errors="coerce")

home = (mt[mt["side"] == "home"]
        .rename(columns={"team_id": "home_team_id", "score": "home_score"})
        [["match_id", "match_date", "home_team_id", "home_score"]])
away = (mt[mt["side"] == "away"]
        .rename(columns={"team_id": "away_team_id", "score": "away_score"})
        [["match_id", "away_team_id", "away_score"]])
matches = home.merge(away, on="match_id", how="inner").dropna(subset=["home_score", "away_score"])

# Option 1 : filtre saison (ratings FC 26 valides ~ 2025-26).
n_before = len(matches)
matches = matches[pd.to_datetime(matches["match_date"]) >= pd.Timestamp(SEASON_CUTOFF)].copy()
print(f"filtre date >= {SEASON_CUTOFF} : {len(matches)}/{n_before} matchs gardés")

def result(row):
    if row["home_score"] > row["away_score"]:
        return "home_win"
    if row["home_score"] < row["away_score"]:
        return "away_win"
    return "draw"

matches["result"] = matches.apply(result, axis=1)

# les deux équipes doivent avoir un rating
matches["home_sofifa_id"] = matches["home_team_id"].map(team_to_sofifa)
matches["away_sofifa_id"] = matches["away_team_id"].map(team_to_sofifa)
rated = matches.dropna(subset=["home_sofifa_id", "away_sofifa_id"]).copy()
rated[["home_sofifa_id", "away_sofifa_id"]] = rated[["home_sofifa_id", "away_sofifa_id"]].astype(int)

print(f"matchs totaux            : {len(matches)}")
print(f"matchs entre 2 équipes notées : {len(rated)}")
print("\nrésultats :")
print(rated["result"].value_counts().to_string())

filtre date >= 2025-07-01 : 947/4052 matchs gardés
matchs totaux            : 947
matchs entre 2 équipes notées : 873

résultats :
result
home_win    417
away_win    274
draw        182


## 4. Assemblage final

Chaque match reçoit les 7 features de l'équipe à domicile (préfixe `home_`) et de l'extérieur (`away_`).

In [5]:
feat_by_id = team_features.set_index("sofifa_team_id")
feat_cols = FEATURES + ["sofifa_team_name"]

def side_features(df, sofifa_col, prefix):
    block = feat_by_id.loc[df[sofifa_col].values, feat_cols].reset_index(drop=True)
    block.columns = [f"{prefix}{c}" for c in block.columns]
    return block

rated = rated.reset_index(drop=True)
dataset = pd.concat([
    rated[["match_id", "match_date", "home_team_id", "away_team_id",
           "home_sofifa_id", "away_sofifa_id"]],
    side_features(rated, "home_sofifa_id", "home_"),
    side_features(rated, "away_sofifa_id", "away_"),
    rated[["home_score", "away_score", "result"]],
], axis=1)

dataset = dataset.sort_values("match_date").reset_index(drop=True)
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
dataset.to_csv(OUTPUT_CSV, index=False)

print("dataset final :", dataset.shape, "->", OUTPUT_CSV)
print("colonnes :", list(dataset.columns))
display(dataset.head(10))

dataset final : (873, 25) -> /Users/safidyramaherison/Documents/Jedha/AIFS/Projects/Final_project/Futbol/github/football-predictor/data/match_outcome_team_ratings.csv
colonnes : ['match_id', 'match_date', 'home_team_id', 'away_team_id', 'home_sofifa_id', 'away_sofifa_id', 'home_overall', 'home_attack', 'home_midfield', 'home_defence', 'home_build_up_style', 'home_defensive_line', 'home_defensive_approach', 'home_sofifa_team_name', 'away_overall', 'away_attack', 'away_midfield', 'away_defence', 'away_build_up_style', 'away_defensive_line', 'away_defensive_approach', 'away_sofifa_team_name', 'home_score', 'away_score', 'result']


,match_id,match_date,home_team_id,away_team_id,home_sofifa_id,away_sofifa_id,home_overall,home_attack,home_midfield,home_defence,home_build_up_style,home_defensive_line,home_defensive_approach,home_sofifa_team_name,away_overall,away_attack,away_midfield,away_defence,away_build_up_style,away_defensive_line,away_defensive_approach,away_sofifa_team_name,home_score,away_score,result
0,4506866,2025-07-01,418,506,243,45,85,88,85,82,Balanced,90,High,Real Madrid,80,81,80,80,Balanced,90,High,Juventus,1,0,home_win
1,4506870,2025-07-05,583,27,73,21,84,85,84,85,Balanced,90,High,Paris Saint-Germain,84,89,85,82,Balanced,65,High,FC Bayern München,2,0,home_win
2,4506871,2025-07-05,418,16,243,22,85,88,85,82,Balanced,90,High,Real Madrid,81,83,80,82,Balanced,65,High,Borussia Dortmund,3,2,home_win
3,4506873,2025-07-09,583,418,73,243,84,85,84,85,Balanced,90,High,Paris Saint-Germain,85,88,85,82,Balanced,90,High,Real Madrid,4,0,home_win
4,4506874,2025-07-13,631,583,5,73,80,80,82,80,Balanced,65,High,Chelsea,84,85,84,85,Balanced,90,High,Paris Saint-Germain,3,0,home_win
5,4624298,2025-07-20,3948,2282,2014,231,74,74,73,75,Balanced,50,Balanced,Union Saint-Gilloise,74,73,76,73,Balanced,90,High,Club Brugge KV,1,2,away_win
6,4625710,2025-07-22,124,265,86,1884,73,72,73,72,Balanced,90,High,Rangers FC,74,73,74,73,Counter,50,Balanced,Panathinaikos FC,2,0,home_win
7,4627422,2025-07-27,2282,1184,231,673,74,73,76,73,Balanced,90,High,Club Brugge KV,73,72,73,75,Balanced,50,Balanced,KRC Genk,2,1,home_win
8,4625727,2025-07-30,265,124,1884,86,74,73,74,73,Counter,50,Balanced,Panathinaikos FC,73,72,73,72,Balanced,90,High,Rangers FC,1,1,draw
9,4683940,2025-07-31,336,294,237,234,79,79,80,77,Balanced,65,High,Sporting CP,78,79,78,77,Balanced,90,High,SL Benfica,0,1,away_win


## 5. Stockage Neon (optionnel)

In [6]:
dataset.to_sql("match_outcome_team_ratings", engine, schema="public", if_exists="replace", index=False)
print("écrit dans public.match_outcome_team_ratings :", dataset.shape)

écrit dans public.match_outcome_team_ratings : (873, 25)


## Notes

- **Aucun navigateur** : les ratings sont re-parsés depuis le HTML local, le reste est SQL. La colonne `defence` est
  désormais la **vraie note** (bug corrigé à la source).
- **Pont fiable** : `team_id` → club modal des joueurs → SoFIFA, en correspondance exacte. Si un `team_id` reste non
  relié (rare), ajoute-le dans `TEAM_OVERRIDES` et ré-exécute.
- **Filtre deux-équipes-notées** : les matchs impliquant un des 9 clubs hors FC 26 sont automatiquement écartés.
- **Filtre saison (`SEASON_CUTOFF`)** : on ne garde que 2025-26 car les ratings sont un instantané FC 26. Élargis la
  date (ex. `"2021-01-01"`) si tu veux plus de volume en assumant des ratings approximatifs pour les vieilles saisons.
- `build_up_style` et `defensive_approach` sont **catégoriels** (à encoder au moment du modèle) ; `defensive_line`
  est numérique.

## 6. Dataset d'entraînement (encodé)

Version prête pour un modèle. Une ligne = un match. Pour **chaque équipe** (domicile / extérieur) :

- **numérique** (gardé tel quel) : `overall`, `attack`, `midfield`, `defence`, `defensive_line` ;
- **catégoriel** : `build_up_style` → **one-hot** (un modèle ne prend pas de chaînes).

On **écarte `defensive_approach`** : c'est un bucket grossier de `defensive_line` (30→Deep, 50→Balanced,
65 **et** 90→High), donc redondant — et moins fin que la valeur numérique qu'on garde.

Les identifiants (`match_id`, `match_date`, noms d'équipe) sont gardés **en référence** — pas des features, à retirer
avant l'entraînement (voir `feature_cols` / `X`). Cible : `result` (`home_win`/`draw`/`away_win`).

In [7]:
num_cols = [
    "home_overall", "home_attack", "home_midfield", "home_defence", "home_defensive_line",
    "away_overall", "away_attack", "away_midfield", "away_defence", "away_defensive_line",
]
cat_cols = ["home_build_up_style", "away_build_up_style"]   # defensive_approach écarté (redondant avec defensive_line)
ref_cols = ["match_id", "match_date", "home_sofifa_team_name", "away_sofifa_team_name"]

# one-hot des catégorielles (0/1)
encoded = pd.get_dummies(dataset[cat_cols], columns=cat_cols, prefix=cat_cols, dtype=int)

# features regroupées : tout le bloc HOME, puis tout le bloc AWAY
all_feats = num_cols + list(encoded.columns)
feature_cols = [c for c in all_feats if c.startswith("home_")] + [c for c in all_feats if c.startswith("away_")]

training_df = pd.concat([dataset[ref_cols + num_cols], encoded, dataset["result"]], axis=1)
training_df = training_df[ref_cols + feature_cols + ["result"]]   # home ensemble, away ensemble

# X / y prêts pour un modèle (les colonnes de référence sont exclues)
X = training_df[feature_cols]
y = training_df["result"]

print("training_df :", training_df.shape, "| X :", X.shape)
print("bloc HOME :", [c for c in feature_cols if c.startswith("home_")])
print("bloc AWAY :", [c for c in feature_cols if c.startswith("away_")])
print("classes :", y.value_counts().to_dict())

TRAIN_CSV = PROJECT_ROOT / "data" / "training_match_dataset.csv"
training_df.to_csv(TRAIN_CSV, index=False)
print("->", TRAIN_CSV)
display(training_df.head())

training_df : (873, 21) | X : (873, 16)
bloc HOME : ['home_overall', 'home_attack', 'home_midfield', 'home_defence', 'home_defensive_line', 'home_build_up_style_Balanced', 'home_build_up_style_Counter', 'home_build_up_style_Short passing']
bloc AWAY : ['away_overall', 'away_attack', 'away_midfield', 'away_defence', 'away_defensive_line', 'away_build_up_style_Balanced', 'away_build_up_style_Counter', 'away_build_up_style_Short passing']
classes : {'home_win': 417, 'away_win': 274, 'draw': 182}
-> /Users/safidyramaherison/Documents/Jedha/AIFS/Projects/Final_project/Futbol/github/football-predictor/data/training_match_dataset.csv


,match_id,match_date,home_sofifa_team_name,away_sofifa_team_name,home_overall,home_attack,home_midfield,home_defence,home_defensive_line,home_build_up_style_Balanced,home_build_up_style_Counter,home_build_up_style_Short passing,away_overall,away_attack,away_midfield,away_defence,away_defensive_line,away_build_up_style_Balanced,away_build_up_style_Counter,away_build_up_style_Short passing,result
0,4506866,2025-07-01,Real Madrid,Juventus,85,88,85,82,90,1,0,0,80,81,80,80,90,1,0,0,home_win
1,4506870,2025-07-05,Paris Saint-Germain,FC Bayern München,84,85,84,85,90,1,0,0,84,89,85,82,65,1,0,0,home_win
2,4506871,2025-07-05,Real Madrid,Borussia Dortmund,85,88,85,82,90,1,0,0,81,83,80,82,65,1,0,0,home_win
3,4506873,2025-07-09,Paris Saint-Germain,Real Madrid,84,85,84,85,90,1,0,0,85,88,85,82,90,1,0,0,home_win
4,4506874,2025-07-13,Chelsea,Paris Saint-Germain,80,80,82,80,65,1,0,0,84,85,84,85,90,1,0,0,home_win


> **Encodage de l'identité des équipes** — si tu veux donner au modèle des « effets équipe » (chaque club comme
> variable), one-hot `home_sofifa_team_name` / `away_sofifa_team_name`. Attention : ~91 clubs × 2 côtés ≈ 180 colonnes
> pour **873 lignes** → fort risque de surapprentissage. À réserver à un modèle régularisé, sinon garde juste les
> features numériques + tactiques ci-dessus.